# 11_Diagnose (fix) — Model hình ảnh có đang học cử chỉ hay học tắt?

**Mục tiêu duy nhất:** kiểm tra xem điểm cao của model hình ảnh ở `10_TrainImage` có phản ánh việc nhận diện cử chỉ tay hay có dấu hiệu dựa vào nguồn video/nền/người quay/phiên quay.

Notebook giữ lại 3 phép thử cũ và làm rõ giới hạn khi diễn giải:
- **A — Đổi nguồn video:** train trên một nguồn (60 FPS hoặc 120 FPS), kiểm tra trên nguồn còn lại; TCN landmark là mốc tham khảo. Chênh lệch lớn là dấu hiệu model khó tổng quát hóa giữa nguồn dữ liệu, nhưng chưa tự chứng minh nguyên nhân là nền.
- **B — 3 seed:** kiểm tra kết quả có ổn định giữa các lần khởi tạo không.
- **C — Trước khi tay xuất hiện:** nếu model đoán đúng trên đoạn không thấy tay, đó là dấu hiệu đáng nghi. Nhưng chỉ kết luận mạnh khi có đủ nhiều video/khung hình.
- **Bổ sung cách đọc:** các metric A/B hiện tại tính theo những chuỗi/khung được tạo từ video, không phải một phép thử can thiệp trực tiếp lên nền. Muốn xác nhận trực tiếp nền có gây ảnh hưởng không, cần ảnh/video gốc để so sánh ảnh gốc với ảnh chỉ giữ tay hoặc che tay/che nền rồi trích xuất lại đặc trưng.

**Không đọc tập test.** Chỉ dùng train/val. Notebook này không sửa `09`, `09b`, `10` và giữ checkpoint trong `Data_321/checkpoints_diag/`.

**Cách chạy:** chạy cell 1 → 8 theo thứ tự. Phép thử C dùng checkpoint `seed0_*` từ B, nên nếu chưa có checkpoint tương ứng thì chạy B trước. Có thể tắt từng phép thử bằng `RUN_A`, `RUN_B`, `RUN_C`.

**Lưu ý:** file này là bản sửa riêng; notebook `11_Diagnose.ipynb` cũ được giữ nguyên. Các output cũ đã được xóa để tránh nhầm output trước sửa với kết quả sau sửa.


In [1]:
#cell 1: thư viện, CUDA, kiểm tra file đặc trưng, chỉ giữ tập train/val, gán nguồn theo fps
import time, math
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
DIRS = {"r18": BASE / "img_feats", "r50": BASE / "img_feats_r50"}
CKPT_DIR = BASE / "checkpoints_diag"
CKPT_DIR.mkdir(exist_ok=True)
assert (DIRS["r18"] / "video_map.csv").exists(), "Chưa có video_map.csv: chạy 09_ImageFeatures.ipynb trước."
vm = pd.read_csv(DIRS["r18"] / "video_map.csv")
for k, d in DIRS.items():
    miss = [f for f in vm.feat if not (d / f).exists()]
    assert not miss, f"Thiếu {len(miss)} file đặc trưng {k} trong {d}: chạy lại 09 hoặc 09b."
vm = vm[vm.tap.isin(["train", "val"])].reset_index(drop=True)      # cố ý bỏ tập test
vm["src"] = np.where(vm.fps < 90, "60fps", "120fps")               # Dataset_hand = 60 fps, dataset_labeled = 120 fps
print("GPU:", torch.cuda.get_device_name(0))
print(vm.groupby(["src", "tap"]).size().unstack())

GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
tap     train  val
src               
120fps     94   19
60fps      57   12


In [2]:
# cell 2: xác định vùng có tay và thống kê đoạn trước khi tay xuất hiện
# Dùng landmark để xác định thời điểm tay xuất hiện; video không có landmark hợp lệ sẽ được loại khỏi phép thử C.
def hand_info(path):
    lm = np.load(BASE / path)["lm"]
    has = ~np.isnan(lm[:, 0, 0])
    idx = np.flatnonzero(has)
    span = np.zeros(len(has), dtype=bool)
    if len(idx):
        span[idx[0]:idx[-1] + 1] = True
        first = int(idx[0])
    else:
        first = None
    return span, first

info = {r.feat: hand_info(r.path) for r in vm.itertuples()}
spans = {k: v[0] for k, v in info.items()}
first_hand = {k: v[1] for k, v in info.items()}
vm["lead_ms"] = [first_hand[f] * 1000 / 120 if first_hand[f] is not None else np.nan for f in vm.feat]
print("Video train/val:", len(vm))
print("Video không có khung nào phát hiện được tay:", sum(v is None for v in first_hand.values()))
print("Số video có đoạn trước khi tay xuất hiện, theo nguồn:")
print(vm.assign(co_pre=vm.lead_ms > 0).groupby(["src", "tap"]).co_pre.agg(["sum", "count"]))
print("Thời gian trước khi thấy tay (ms), theo nguồn; chỉ tính video có phát hiện tay:")
print(vm.dropna(subset=["lead_ms"]).groupby("src").lead_ms.describe()[["min", "50%", "max"]].round(0))


Video train/val: 182
Video không có khung nào phát hiện được tay: 0
Số video có đoạn trước khi tay xuất hiện, theo nguồn:
              sum  count
src    tap              
120fps train    0     94
       val      0     19
60fps  train    1     57
       val      3     12
Thời gian trước khi thấy tay (ms), theo nguồn; chỉ tính video có phát hiện tay:
        min  50%    max
src                    
120fps  0.0  0.0    0.0
60fps   0.0  0.0  167.0


In [3]:
#cell 3: dựng dữ liệu — chuỗi 30 fps từ đặc trưng hình ảnh, và chuỗi 120 fps từ landmark (cho TCN landmark làm mốc)
def pack(seqs, dtype):
    n, tmax, dim = len(seqs), max(len(s[0]) for s in seqs), seqs[0][0].shape[1]
    X, M, Tm = np.zeros((n, tmax, dim), dtype), np.zeros((n, tmax), bool), np.zeros((n, tmax), np.float32)
    for i, (x, mk, _, t) in enumerate(seqs):
        X[i, :len(x)], M[i, :len(x)], Tm[i, :len(x)] = x, mk, t
    to = lambda a: torch.tensor(a, device=dev)
    return to(X), to(np.array([s[2] for s in seqs])), to(M), to(Tm)   # đặc trưng, nhãn, khung tính, thời gian (ms)

def build_img(bk, df, versions, part="span"):
    """part='span': khung trong vùng có tay (như 10_TrainImage); part='pre': khung trước khi thấy tay."""
    seqs = []
    for r in df.itertuples():
        f = np.load(DIRS[bk] / r.feat)["f"]                           # (4, T, D) float16
        T, stride, sp = f.shape[1], int(round(r.fps / 30)), spans[r.feat]
        j = np.arange(T)
        i120 = np.minimum(np.rint(j * 120 / r.fps).astype(int), len(sp) - 1)
        first = first_hand[r.feat]
        if part == "pre" and first is None:
            continue
        mask = sp[i120] if part == "span" else (i120 < first)
        if not mask.any():
            continue
        for v in versions:
            for o in range(stride):
                seqs.append((f[v, o::stride], mask[o::stride], r.nhan, j[o::stride] * 1000 / r.fps))
    return pack(seqs, np.float16) if seqs else None

def feat_lm(path):                                                    # giống hàm feat() ở 04_Trainv2
    lm = np.load(BASE / path)["lm"]
    lm = lm - lm[:, :1]
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), ~np.isnan(pos[:, 0])

def build_lm(df):
    seqs = []
    for r in df.itertuples():
        x, has = feat_lm(r.path)
        if has.any():
            seqs.append((x, has, r.nhan, np.arange(len(x)) * 1000 / 120))   # khung tính = khung MediaPipe thấy tay
    return pack(seqs, np.float32)

print("Đã định nghĩa build_img() và build_lm().")

Đã định nghĩa build_img() và build_lm().


In [4]:
#cell 4: model — TCN và GRU cho đặc trưng hình ảnh (như 10_TrainImage), TCN landmark (như 04/07) và augmentation landmark wide45_lownoise
class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, d_in, c=64, p=0.3, dil=(1, 2, 4)):
        super().__init__()
        s.inp = nn.Sequential(nn.Dropout(p), nn.Linear(d_in, c))
        s.blocks = nn.Sequential(*[Block(c, d) for d in dil]); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

class GRU(nn.Module):
    def __init__(s, d_in, c=64, p=0.3):
        super().__init__()
        s.inp = nn.Sequential(nn.Dropout(p), nn.Linear(d_in, c)); s.rnn = nn.GRU(c, c, batch_first=True); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.rnn(F.relu(s.inp(x)))[0])

make_img = lambda name, d: (lambda: {"TCN": TCN, "GRU": GRU}[name](d))
make_lm = lambda: TCN(126, p=0.0, dil=(1, 2, 4, 8, 16))             # như TCN landmark ở 04 (tầm nhìn 63 khung ở 120 fps)

def lm_aug(x, hand_mask, rot=(40, 45, 40), noise=0.005):            # augmentation của bản wide45_lownoise (07_TrainWide)
    b, T = x.shape[:2]
    pos = x[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    ax, ay, az = [(torch.rand(b, 1, 1, device=x.device) * 2 - 1) * math.radians(d) for d in rot]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    flip = torch.rand(b, 1, 1, device=x.device) < 0.5
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])
    pos = pos + torch.randn_like(pos) * noise * valid
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None, None]
    return torch.cat([pos.reshape(b, T, 63), vel.reshape(b, T, 63)], dim=-1)

print("Đã định nghĩa model và augmentation.")

Đã định nghĩa model và augmentation.


In [5]:
#cell 5: train (hoặc nạp checkpoint) và các chỉ số
CFG_VER = 1

def loss_acc(net, X, Y, M):
    out = net(X.float())
    ce = F.cross_entropy(out.transpose(1, 2), Y[:, None].expand(-1, X.shape[1]), reduction="none")
    return (ce * M).sum() / M.sum(), ((out.argmax(-1) == Y[:, None]) & M).sum() / M.sum()

def get_net(tag, make, tr=None, va=None, aug=None, seed=0, epochs=60, bs=16, patience=10):
    cfg = {"tag": tag, "ver": CFG_VER, "seed": seed, "aug": aug is not None}
    path = CKPT_DIR / f"{tag}.pt"
    torch.manual_seed(seed)
    net = make().to(dev)
    if path.exists():
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            net.load_state_dict(ck["w"])
            print(f"{tag}: nạp checkpoint | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print(f"{tag}: checkpoint cũ có cấu hình khác, train lại.")
    assert tr is not None and va is not None, f"Chưa có checkpoint {tag}: chạy phép thử tạo ra nó trước."

    (Xt, Yt, Mt, _), (Xv, Yv, Mv, _) = tr, va
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    best, wait, t0 = float("inf"), 0, time.time()
    for ep in range(1, epochs + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev)
        for i in range(0, len(Xt), bs):
            ids = perm[i:i + bs]
            xb = Xt[ids].float()
            if aug is not None:
                xb = aug(xb, Mt[ids])
            loss, _ = loss_acc(net, xb, Yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl, _ = loss_acc(net, Xv, Yv, Mv)
        if vl.item() < best:
            best, wait = vl.item(), 0
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "cfg": cfg}, path)
        else:
            wait += 1
            if wait >= patience:
                break
    ck = torch.load(path, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"{tag}: train xong | tốt nhất epoch {ck['epoch']} | val_loss {best:.4f} | {time.time() - t0:.0f}s | lưu ở {path}")
    return net.eval()

@torch.no_grad()
def val_metrics(net, X, Y, M, Tm):                                  # giống 10_TrainImage
    conf, pred = net(X.float()).softmax(-1).max(-1)
    ok = (pred == Y[:, None]) & M
    half = M & (M.cumsum(1) <= M.sum(1, keepdim=True) / 2)
    hits = [(ok & (conf >= t)).any(1).float().mean().item() for t in (0.5, 0.6, 0.7, 0.8, 0.9)]
    return {"val_acc": ok.sum().item() / M.sum().item(), "acc_nửa_đầu": (ok & half).sum().item() / max(half.sum().item(), 1),
            "hit_TB": float(np.mean(hits))}

@torch.no_grad()
def frame_acc(net, X, Y, M):
    return ((net(X.float()).argmax(-1) == Y[:, None]) & M).sum().item() / M.sum().item()

print("Đã định nghĩa get_net(), val_metrics(), frame_acc().")

Đã định nghĩa get_net(), val_metrics(), frame_acc().


In [6]:
#cell 6: phép thử A — train trên một nguồn (train), dừng sớm bằng val cùng nguồn, thử trên toàn bộ video nguồn kia (không dùng test)
RUN_A = True
JOBS = {"r18": ("TCN", "GRU"), "r50": ("TCN", "GRU"), "lm": ("TCN",)}      # lm = TCN landmark làm mốc

rows = []
if RUN_A:
    for a, b in [("60fps", "120fps"), ("120fps", "60fps")]:
        tr_df = vm[(vm.src == a) & (vm.tap == "train")]
        va_df = vm[(vm.src == a) & (vm.tap == "val")]
        ev_df = vm[vm.src == b]
        print(f"\nA: train {a} (train={len(tr_df)}, val={len(va_df)}) → thử {b} (video={len(ev_df)})")
        for bk, names in JOBS.items():
            if bk == "lm":
                tr, va, ev, aug = build_lm(tr_df), build_lm(va_df), build_lm(ev_df), lm_aug
            else:
                tr, va, ev, aug = build_img(bk, tr_df, [0, 1, 2, 3]), build_img(bk, va_df, [0]), build_img(bk, ev_df, [0]), None
            for name in names:
                make = make_lm if bk == "lm" else make_img(name, tr[0].shape[2])
                net = get_net(f"x{a}_to_{b}_{bk}_{name}", make, tr, va, aug)
                m_in, m_x = val_metrics(net, *va), val_metrics(net, *ev)
                rows.append({"train→thử": f"{a}→{b}", "backbone": bk, "model": name,
                             "in_acc": m_in["val_acc"], "cross_acc": m_x["val_acc"], "tụt": m_in["val_acc"] - m_x["val_acc"],
                             "cross_nửa_đầu": m_x["acc_nửa_đầu"], "cross_hit_TB": m_x["hit_TB"]})
            del tr, va, ev
            torch.cuda.empty_cache()
    res_a = pd.DataFrame(rows)
    display(res_a.round(3))
    print("Cách đọc: cột 'tụt' lớn nghĩa là accuracy giảm khi đổi nguồn. Đây là dấu hiệu kém tổng quát hóa giữa hai nguồn, chưa đủ để kết luận riêng nền/người quay là nguyên nhân.")


A: train 60fps (train=57, val=12) → thử 120fps (video=113)
x60fps_to_120fps_r18_TCN: nạp checkpoint | epoch 10 | val_loss 0.3077
x60fps_to_120fps_r18_GRU: nạp checkpoint | epoch 18 | val_loss 0.3020
x60fps_to_120fps_r50_TCN: nạp checkpoint | epoch 20 | val_loss 0.2211
x60fps_to_120fps_r50_GRU: nạp checkpoint | epoch 8 | val_loss 0.3017
x60fps_to_120fps_lm_TCN: nạp checkpoint | epoch 30 | val_loss 0.4809

A: train 120fps (train=94, val=19) → thử 60fps (video=69)
x120fps_to_60fps_r18_TCN: nạp checkpoint | epoch 2 | val_loss 0.3177
x120fps_to_60fps_r18_GRU: nạp checkpoint | epoch 1 | val_loss 0.4387
x120fps_to_60fps_r50_TCN: nạp checkpoint | epoch 1 | val_loss 0.4850
x120fps_to_60fps_r50_GRU: nạp checkpoint | epoch 2 | val_loss 0.3188
x120fps_to_60fps_lm_TCN: nạp checkpoint | epoch 50 | val_loss 0.0203


,train→thử,backbone,model,in_acc,cross_acc,tụt,cross_nửa_đầu,cross_hit_TB
0,60fps→120fps,r18,TCN,0.844,0.716,0.127,0.728,0.764
1,60fps→120fps,r18,GRU,0.838,0.674,0.164,0.691,0.716
2,60fps→120fps,r50,TCN,0.904,0.639,0.265,0.694,0.751
3,60fps→120fps,r50,GRU,0.882,0.719,0.164,0.778,0.770
4,60fps→120fps,lm,TCN,0.766,0.766,-0.000,0.776,0.835
5,120fps→60fps,r18,TCN,0.848,0.414,0.433,0.338,0.535
6,120fps→60fps,r18,GRU,0.834,0.467,0.367,0.355,0.603
7,120fps→60fps,r50,TCN,0.855,0.387,0.468,0.352,0.497
8,120fps→60fps,r50,GRU,0.885,0.486,0.399,0.418,0.580
9,120fps→60fps,lm,TCN,0.999,0.675,0.323,0.431,0.959


Cách đọc: cột 'tụt' lớn nghĩa là accuracy giảm khi đổi nguồn. Đây là dấu hiệu kém tổng quát hóa giữa hai nguồn, chưa đủ để kết luận riêng nền/người quay là nguyên nhân.


In [7]:
#cell 7: phép thử B — train lại với nhiều hạt giống (train → val như 10_TrainImage), xem độ dao động
RUN_B = True
SEEDS = [0, 1, 2]

if RUN_B:
    rows_b = []
    for bk in DIRS:
        tr, va = build_img(bk, vm[vm.tap == "train"], [0, 1, 2, 3]), build_img(bk, vm[vm.tap == "val"], [0])
        for name in ("TCN", "GRU"):
            for s in SEEDS:
                net = get_net(f"seed{s}_{bk}_{name}", make_img(name, tr[0].shape[2]), tr, va, seed=s)
                rows_b.append({"backbone": bk, "model": name, "seed": s, **val_metrics(net, *va)})
        del tr, va
        torch.cuda.empty_cache()
    res_b = pd.DataFrame(rows_b)
    display(res_b.round(3))
    print("Trung bình và độ lệch chuẩn theo các hạt giống:")
    display(res_b.groupby(["backbone", "model"])[["val_acc", "acc_nửa_đầu", "hit_TB"]].agg(["mean", "std"]).round(3))

seed0_r18_TCN: nạp checkpoint | epoch 9 | val_loss 0.2287
seed1_r18_TCN: nạp checkpoint | epoch 6 | val_loss 0.3248
seed2_r18_TCN: nạp checkpoint | epoch 25 | val_loss 0.4253
seed0_r18_GRU: nạp checkpoint | epoch 5 | val_loss 0.4472
seed1_r18_GRU: nạp checkpoint | epoch 11 | val_loss 0.1255
seed2_r18_GRU: nạp checkpoint | epoch 4 | val_loss 0.6395
seed0_r50_TCN: nạp checkpoint | epoch 1 | val_loss 0.3495
seed1_r50_TCN: nạp checkpoint | epoch 1 | val_loss 0.4125
seed2_r50_TCN: nạp checkpoint | epoch 1 | val_loss 0.6641
seed0_r50_GRU: nạp checkpoint | epoch 10 | val_loss 0.0645
seed1_r50_GRU: nạp checkpoint | epoch 13 | val_loss 0.0857
seed2_r50_GRU: nạp checkpoint | epoch 9 | val_loss 0.0791


,backbone,model,seed,val_acc,acc_nửa_đầu,hit_TB
0,r18,TCN,0,0.870,0.841,0.978
1,r18,TCN,1,0.859,0.825,0.960
2,r18,TCN,2,0.862,0.839,0.976
3,r18,GRU,0,0.830,0.784,0.896
4,r18,GRU,1,0.944,0.899,1.000
5,r18,GRU,2,0.859,0.831,0.920
6,r50,TCN,0,0.853,0.818,0.938
7,r50,TCN,1,0.895,0.842,0.938
8,r50,TCN,2,0.868,0.848,0.940
9,r50,GRU,0,0.970,0.942,1.000


Trung bình và độ lệch chuẩn theo các hạt giống:


val_acc        acc_nửa_đầu        hit_TB       
                  mean    std        mean    std   mean    std
backbone model                                                
r18      GRU     0.877  0.059       0.838  0.058  0.939  0.054
         TCN     0.864  0.006       0.835  0.009  0.971  0.010
r50      GRU     0.966  0.004       0.935  0.007  1.000  0.000
         TCN     0.872  0.021       0.836  0.016  0.939  0.001

In [8]:
# cell 8: phép thử C — độ chính xác trên đoạn trước khi tay xuất hiện
RUN_C = True

if RUN_C:
    va_df = vm[vm.tap == "val"]
    pre_df = va_df[va_df.lead_ms > 0].copy()
    print(f"Video val có đoạn trước khi thấy tay: {len(pre_df)}/{len(va_df)}")
    rows_c = []
    if len(pre_df) == 0:
        print("Không có đoạn trước khi tay xuất hiện; phép thử C không thể kết luận với tập val hiện tại.")
    else:
        for bk in DIRS:
            pre = build_img(bk, pre_df, [0], part="pre")
            if pre is None:
                print(f"{bk}: không tạo được chuỗi pre-hand.")
                continue
            n_frames = int(pre[2].sum().item())
            for name in ("TCN", "GRU"):
                tag = f"seed0_{bk}_{name}"
                path = CKPT_DIR / f"{tag}.pt"
                if not path.exists():
                    print(f"Bỏ qua {tag}: chưa có checkpoint. Hãy chạy phép thử B trước.")
                    continue
                net = get_net(tag, make_img(name, pre[0].shape[2]))
                rows_c.append({"backbone": bk, "model": name,
                               "video val có đoạn pre": len(pre_df),
                               "số khung pre được chấm": n_frames,
                               "acc trước khi thấy tay": frame_acc(net, pre[0], pre[1], pre[2])})
        if rows_c:
            res_c = pd.DataFrame(rows_c)
            display(res_c.round(3))
            if len(pre_df) < 10 or int(res_c["số khung pre được chấm"].max()) < 100:
                print("CẢNH BÁO: số video/khung pre-hand còn ít; kết quả chỉ là dấu hiệu sơ bộ, không nên dùng làm kết luận cuối cùng.")
            print("Mốc ngẫu nhiên xấp xỉ 0.333 chỉ có ý nghĩa khi ba lớp cân bằng. Hãy xem cả số video và số khung, không chỉ accuracy.")


Video val có đoạn trước khi thấy tay: 3/31
seed0_r18_TCN: nạp checkpoint | epoch 9 | val_loss 0.2287
seed0_r18_GRU: nạp checkpoint | epoch 5 | val_loss 0.4472
seed0_r50_TCN: nạp checkpoint | epoch 1 | val_loss 0.3495
seed0_r50_GRU: nạp checkpoint | epoch 10 | val_loss 0.0645


,backbone,model,video val có đoạn pre,số khung pre được chấm,acc trước khi thấy tay
0,r18,TCN,3,16,0.938
1,r18,GRU,3,16,0.375
2,r50,TCN,3,16,1.000
3,r50,GRU,3,16,1.000


CẢNH BÁO: số video/khung pre-hand còn ít; kết quả chỉ là dấu hiệu sơ bộ, không nên dùng làm kết luận cuối cùng.
Mốc ngẫu nhiên xấp xỉ 0.333 chỉ có ý nghĩa khi ba lớp cân bằng. Hãy xem cả số video và số khung, không chỉ accuracy.


## Cách đọc kết quả
- **A (quan trọng nhất):** so cột `tụt` của model hình ảnh với dòng `lm` cùng hướng. Hình ảnh tụt mạnh còn `lm` ít tụt nghĩa là học tắt theo nền/người. Cả hai cùng tụt nhiều nghĩa là đổi nguồn vốn khó với mọi model, cần dữ liệu người và camera mới. Hình ảnh giữ gần mức cũ là dấu hiệu tốt.
- **B:** nếu độ lệch chuẩn (`std`) lớn hơn khoảng cách giữa các model thì thứ hạng ở `10_TrainImage` không đáng tin; chỉ nên báo cáo giá trị trung bình.
- **C:** `acc trước khi thấy tay` cao hơn nhiều so với 0.333 nghĩa là model đoán đúng cả khi chưa có tay, tức đang nhìn nền. Nếu mọi video đều có tay ngay khung đầu thì phép thử này không có ý nghĩa, xem thống kê ở cell 2.
- Các số ở đây dựa trên tập val nhỏ (khoảng 31 video) nên chênh lệch vài điểm phần trăm chưa đáng tin.

## Kết luận sau khi chạy — chỉ kết luận theo bằng chứng có sẵn

- **Dấu hiệu nghi học tắt/khó tổng quát hóa:** accuracy giảm rõ khi train một nguồn và thử nguồn khác ở phép thử A.
- **Độ ổn định:** phép thử B cho biết các seed có cho kết quả tương tự nhau không; ổn định không đồng nghĩa model chắc chắn nhìn đúng bàn tay.
- **Bằng chứng trước khi thấy tay:** nếu accuracy ở C cao hơn mức ngẫu nhiên, model có thể đang khai thác thông tin ngoài cử chỉ. Nếu số video/khung ít, coi đây chỉ là dấu hiệu sơ bộ.
- **Giới hạn quan trọng:** A thay đổi cùng lúc nhiều yếu tố (nguồn dữ liệu, nền, người, ánh sáng, cách quay và có thể cả phân bố nhãn). C chỉ dùng các đoạn ngắn không thấy tay. Vì vậy, các phép thử này **không đủ để khẳng định nguyên nhân là nền**.
- **Bước xác nhận trực tiếp tiếp theo:** cần video/ảnh gốc và trích xuất lại đặc trưng trong các điều kiện đối chứng: (1) ảnh gốc, (2) chỉ giữ vùng bàn tay/cắt nền, (3) che bàn tay, (4) giữ nền nhưng che bàn tay. So sánh cùng tập video và cùng nhãn. Nếu che tay làm kết quả giảm mạnh còn thay/che nền ít ảnh hưởng, đó là bằng chứng tốt hơn model dựa vào cử chỉ.
